In [1]:
# !pip install pandas sentence-transformers scikit-learn


In [2]:

from google.colab import drive

drive.mount('/content/drive')
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'

Mounted at /content/drive


In [3]:
import os
import pickle
import json
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

# model="bert-base-uncased"

def prepare_embedding_text(df, title_col='title', content_col='content', max_words=512):
    print("Preparing and truncating text...")
    df_prep = df.copy()
    
    # # Safely extract and fill NaN values to avoid concatenation errors
    # t_col = df_prep[title_col].fillna('') if title_col in df_prep.columns else pd.Series(['']*len(df_prep))
    # c_col = df_prep[content_col].fillna('') if content_col in df_prep.columns else pd.Series(['']*len(df_prep))
    
    # # Combine title and content, then truncate by word count
    # df_prep['raw_text'] = t_col.astype(str) + " " + c_col.astype(str)
    df_prep['safe_content'] = df_prep[content_col].apply(lambda x: ' '.join(x.split()[:max_words]))
    return df_prep

def generate_and_save_embeddings(text_list, checkpoint_dir="checkpoints_5_4", model_name='bert-base-uncased', batch_size=32):
    print("Initializing embedding model...")
    # Automatically uses GPU in Colab if Hardware Accelerator is set to T4 GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    model = SentenceTransformer(model_name, device=device)
    
    print(f"Generating embeddings for {len(text_list)} items...")
    embeddings = model.encode(text_list, show_progress_bar=True, batch_size=batch_size)
    
    print(f"Saving checkpoints to {checkpoint_dir}...")
    os.makedirs(checkpoint_dir, exist_ok=True)
    with open(os.path.join(checkpoint_dir, 'embeddings_ckpt.pkl'), 'wb') as f:
        pickle.dump(embeddings, f)
        
    return embeddings

def save_final_features(df, embeddings, output_dir="data", dataset_prefix="processed_v1"):
    print("Splitting and saving final features...")
    df_final = df.copy()
    df_final['embeddings'] = list(embeddings)
    
    # 80/10/10 Train, Validation, Test Split
    train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    os.makedirs(output_dir, exist_ok=True)
    train_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_train.pkl"))
    val_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_val.pkl"))
    test_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_test.pkl"))
    df_final.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_full.pkl"))
    
    print("Saved train/val/test splits successfully!")
    return df_final

In [4]:
import pandas as pd

df_posts = pd.read_csv(f"{DESTINATION_DIR}/reddit_22_3.csv")

In [5]:
df_posts

,score,title,selftext,forum,created_utc_dt,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment,hour,...,ttr,hapax,stopword_ratio,burstiness,punctuation_density,hedging_score,self_reference_rate,forum_philosophy,forum_technology,forum_todayilearned
0,0,Proof in support of determinism tell me what y...,1) Peoples choices are determined by their exp...,philosophy,2009-06-19 04:30:01+00:00,1.0,0.283940,0.9604,-0.5515,4,...,0.555556,0.333333,0.519231,0.473014,0.023932,0.000000,0.009615,1.0,0.0,0.0
1,34,Did Jesus commit suicide?,I am in a philosophy class and I have to write...,philosophy,2009-06-20 17:19:15+00:00,1.0,-0.248040,0.5023,-0.9345,17,...,0.540741,0.340741,0.490196,0.550535,0.026022,0.000000,0.071895,1.0,0.0,0.0
2,19,We are a bunch of stupid apes.,People give up their lives for no reason. We ...,philosophy,2009-06-21 00:36:03+00:00,1.0,0.045660,0.8900,-0.9476,0,...,0.479893,0.324397,0.525164,0.627089,0.032870,0.218818,0.067834,1.0,0.0,0.0
3,13,What does it mean to be 'Interesting'?,Who do you consider to be interesting? I have ...,philosophy,2009-06-22 12:54:51+00:00,1.0,0.324690,0.9711,-0.3182,12,...,0.653846,0.403846,0.644068,0.413087,0.028846,0.000000,0.033898,1.0,0.0,0.0
4,0,What's your theory of human nature?,Are all people basically selfish? Are they al...,philosophy,2009-06-22 20:05:23+00:00,0.6,-0.122800,0.2500,-0.9868,20,...,0.722222,0.500000,0.470588,0.663560,0.036408,0.000000,0.044118,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34862,184,When musicians started selling singles for $.9...,If you could legally stream just one or two ch...,technology,2019-03-27 20:00:54+00:00,1.0,0.266050,0.9574,-0.4019,20,...,0.888889,0.822222,0.415094,0.330593,0.028302,1.886792,0.018868,0.0,1.0,0.0
34863,2,There's concern around Hauwei's big business n...,I am seeing massive uptake of Hauwei's handset...,technology,2019-03-28 09:27:33+00:00,0.1,0.949600,0.9496,0.9496,9,...,0.803030,0.666667,0.389610,0.202031,0.020964,0.000000,0.051948,0.0,1.0,0.0
34864,10,Got a tech question or want to discuss tech? W...,"##Greetings Good People of /r/Technology,\n\nW...",technology,2019-03-30 00:05:14+00:00,1.0,0.380670,0.9390,0.0000,0,...,0.723684,0.565789,0.353535,0.466905,0.089664,0.000000,0.010101,0.0,1.0,0.0
34865,0,What happened to StackOverflow?,Few hours ago it was fine. And now [this. (Sta...,technology,2019-03-31 11:56:00+00:00,0.8,-0.112138,0.2714,-0.4484,11,...,1.000000,1.000000,0.500000,0.515079,0.119658,0.000000,0.000000,0.0,1.0,0.0


In [6]:
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')

In [7]:



# Make sure to manually upload 'moltbook_3month_2026_data_all_7.json' to the Colab file explorer first



# 1. Truncate and prep text
df_prep = prepare_embedding_text(df_posts,  content_col="content")

# 2. Extract BERT Embeddings
embeddings = generate_and_save_embeddings(df_prep['safe_content'].tolist(), checkpoint_dir=f"{DESTINATION_DIR}/checkpoints_reddit_11_4", model_name='bert-base-uncased', batch_size=32)

# 3. Save Final Features to Pickle (includes automatic train/val/test splits!)
final_df = save_final_features(df_prep, embeddings, output_dir=DESTINATION_DIR, dataset_prefix="reddit_11_4")

Preparing and truncating text...
Initializing embedding model...
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Generating embeddings for 34867 items...


Batches:   0%|          | 0/1090 [00:00<?, ?it/s]

Saving checkpoints to /content/drive/MyDrive/Colab Notebooks/data/cds/checkpoints_reddit_11_4...
Splitting and saving final features...
Saved train/val/test splits successfully!


In [8]:
final_df

,score,title,selftext,forum,created_utc_dt,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment,hour,...,stopword_ratio,burstiness,punctuation_density,hedging_score,self_reference_rate,forum_philosophy,forum_technology,forum_todayilearned,safe_content,embeddings
0,0,Proof in support of determinism tell me what y...,1) Peoples choices are determined by their exp...,philosophy,2009-06-19 04:30:01+00:00,1.0,0.283940,0.9604,-0.5515,4,...,0.519231,0.473014,0.023932,0.000000,0.009615,1.0,0.0,0.0,Proof in support of determinism tell me what y...,"[-0.037368815, 0.2573366, 0.17375503, -0.00970..."
1,34,Did Jesus commit suicide?,I am in a philosophy class and I have to write...,philosophy,2009-06-20 17:19:15+00:00,1.0,-0.248040,0.5023,-0.9345,17,...,0.490196,0.550535,0.026022,0.000000,0.071895,1.0,0.0,0.0,Did Jesus commit suicide? I am in a philosophy...,"[0.0026358643, 0.11318106, -0.02474467, -0.234..."
2,19,We are a bunch of stupid apes.,People give up their lives for no reason. We ...,philosophy,2009-06-21 00:36:03+00:00,1.0,0.045660,0.8900,-0.9476,0,...,0.525164,0.627089,0.032870,0.218818,0.067834,1.0,0.0,0.0,We are a bunch of stupid apes. People give up ...,"[0.18968612, 0.13119088, 0.24845333, -0.026331..."
3,13,What does it mean to be 'Interesting'?,Who do you consider to be interesting? I have ...,philosophy,2009-06-22 12:54:51+00:00,1.0,0.324690,0.9711,-0.3182,12,...,0.644068,0.413087,0.028846,0.000000,0.033898,1.0,0.0,0.0,What does it mean to be 'Interesting'? Who do ...,"[0.18006337, 0.27622178, 0.054709073, -0.22981..."
4,0,What's your theory of human nature?,Are all people basically selfish? Are they al...,philosophy,2009-06-22 20:05:23+00:00,0.6,-0.122800,0.2500,-0.9868,20,...,0.470588,0.663560,0.036408,0.000000,0.044118,1.0,0.0,0.0,What's your theory of human nature? Are all pe...,"[0.37292367, 0.1502915, 0.08200261, 0.09410036..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34862,184,When musicians started selling singles for $.9...,If you could legally stream just one or two ch...,technology,2019-03-27 20:00:54+00:00,1.0,0.266050,0.9574,-0.4019,20,...,0.415094,0.330593,0.028302,1.886792,0.018868,0.0,1.0,0.0,When musicians started selling singles for $.9...,"[0.38777104, -0.13692825, 0.21760793, 0.166016..."
34863,2,There's concern around Hauwei's big business n...,I am seeing massive uptake of Hauwei's handset...,technology,2019-03-28 09:27:33+00:00,0.1,0.949600,0.9496,0.9496,9,...,0.389610,0.202031,0.020964,0.000000,0.051948,0.0,1.0,0.0,There's concern around Hauwei's big business n...,"[0.03419477, 0.17571296, 0.35805562, 0.0782693..."
34864,10,Got a tech question or want to discuss tech? W...,"##Greetings Good People of /r/Technology,\n\nW...",technology,2019-03-30 00:05:14+00:00,1.0,0.380670,0.9390,0.0000,0,...,0.353535,0.466905,0.089664,0.000000,0.010101,0.0,1.0,0.0,Got a tech question or want to discuss tech? W...,"[0.22531416, -0.042021666, 0.45746735, -0.2180..."
34865,0,What happened to StackOverflow?,Few hours ago it was fine. And now [this. (Sta...,technology,2019-03-31 11:56:00+00:00,0.8,-0.112138,0.2714,-0.4484,11,...,0.500000,0.515079,0.119658,0.000000,0.000000,0.0,1.0,0.0,What happened to StackOverflow? Few hours ago ...,"[-0.032352444, -0.25849077, 0.39427316, -0.189..."
